<a href="https://colab.research.google.com/github/Timmythaw/langgraph-adk-edu-comparison/blob/main/notebooks/03_llm_judge.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🏗️ Notebook 03 — LLM-as-a-Judge

> **Fully self-contained.** This notebook installs all dependencies, builds both the LangGraph and Google ADK teacher-assistant systems from scratch, runs all 3 evaluation scenarios on both frameworks, then uses **Gemini 2.5 Pro** as a judge to score every response.

**Colab Secrets required:**
- `GOOGLE_CLOUD_PROJECT`
- `GOOGLE_CLOUD_LOCATION` (Vertex AI Search location, e.g. `global`)
- `VERTEX_AI_SEARCH_DATASTORE_ID`
- `LANGSMITH_API_KEY`

| Section | What happens |
|---|---|
| 1 | Install all dependencies |
| 2 | Secrets, auth, Vertex AI init |
| 3 | Shared Vertex AI Search tool |
| 4 | Build LangGraph system (full clone of NB01) |
| 5 | Build Google ADK system (full clone of NB02) |
| 6 | Runners for both frameworks |
| 7 | Run evaluation scenarios |
| 8 | LLM Judge (Gemini 2.5 Pro, thinking=2048) |
| 9 | Score all responses |
| 10 | Aggregate + summary tables |
| 11 | Visualisations (bar, radar, scatter) |
| 12 | LLM final verdict |
| 13 | Export CSV + Markdown report |

## 1 — Install Dependencies

In [1]:
!pip install langgraph langchain-google-genai langchain-google-community \
    google-adk google-cloud-aiplatform google-cloud-discoveryengine \
    google-api-python-client google-auth-httplib2 google-auth-oauthlib \
    langsmith pandas matplotlib -q
print("All packages installed.")

All packages installed.


## 2 — Secrets, Auth & Config

In [2]:
import os
from google.colab import userdata, auth

PROJECT_ID   = userdata.get("GOOGLE_CLOUD_PROJECT")
LOCATION     = userdata.get("GOOGLE_CLOUD_LOCATION")
DATASTORE_ID = userdata.get("VERTEX_AI_SEARCH_DATASTORE_ID")

os.environ["GOOGLE_CLOUD_PROJECT"]      = PROJECT_ID
os.environ["GOOGLE_CLOUD_LOCATION"]     = LOCATION
os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "1"
os.environ["LANGCHAIN_TRACING_V2"]      = "true"
os.environ["LANGCHAIN_API_KEY"]         = userdata.get("LANGSMITH_API_KEY")
os.environ["LANGCHAIN_PROJECT"]         = "langgraph-adk-edu-comparison"
os.environ["LANGCHAIN_ENDPOINT"]        = "https://api.smith.langchain.com"

auth.authenticate_user()

import vertexai
from google.cloud import aiplatform
vertexai.init(project=PROJECT_ID, location="us-central1")
aiplatform.init(project=PROJECT_ID, location="us-central1")

print("Project  :", PROJECT_ID[:20])
print("Location :", LOCATION)
print("Datastore:", DATASTORE_ID[:20])
print("LangSmith: enabled ->" , os.environ["LANGCHAIN_PROJECT"])

Project  : edu-teacher-assistan
Location : global
Datastore: curriculum-connector
LangSmith: enabled -> langgraph-adk-edu-comparison


In [3]:
from langsmith.integrations.google_adk import configure_google_adk
configure_google_adk()
print("ADK LangSmith integration configured.")

/usr/local/lib/python3.12/dist-packages/google/adk/features/_feature_decorator.py:72: UserWarning: [EXPERIMENTAL] feature FeatureName.PLUGGABLE_AUTH is enabled.
  check_feature_enabled()


ADK LangSmith integration configured.


## 3 — Shared Tool: Vertex AI Search

In [4]:
from google.cloud import discoveryengine_v1 as discoveryengine

def retrieve_course_materials(query: str, page_size: int = 5) -> str:
    """Search curriculum datastore for relevant course materials.

    Args:
        query: Natural language search query.
        page_size: Number of results to retrieve.

    Returns:
        Concatenated snippet text, or 'No relevant materials found.'
    """
    client = discoveryengine.SearchServiceClient()
    serving_config = (
        f"projects/{PROJECT_ID}/locations/{LOCATION}"
        f"/collections/default_collection/dataStores/{DATASTORE_ID}"
        f"/servingConfigs/default_config"
    )
    request = discoveryengine.SearchRequest(
        serving_config=serving_config, query=query, page_size=page_size,
        content_search_spec=discoveryengine.SearchRequest.ContentSearchSpec(
            snippet_spec=discoveryengine.SearchRequest.ContentSearchSpec.SnippetSpec(
                return_snippet=True
            ),
            summary_spec=discoveryengine.SearchRequest.ContentSearchSpec.SummarySpec(
                summary_result_count=min(page_size, 5), include_citations=True,
            ),
        ),
    )
    response = client.search(request)
    snippets = []
    for result in response.results:
        doc = result.document
        if doc.derived_struct_data:
            for s in doc.derived_struct_data.get("snippets", []):
                text = s.get("snippet", "").strip()
                if text:
                    snippets.append(text)
    return "\n\n---\n\n".join(snippets) if snippets else "No relevant materials found."

print("Vertex AI Search tool ready.")

Vertex AI Search tool ready.


## 4 — Build LangGraph System

Complete replica of `01_langgraph_system.ipynb` with HITL auto-approved for evaluation.

In [5]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import HumanMessage, AIMessage
from typing import TypedDict, Annotated, Literal
import operator

class TeacherState(TypedDict):
    messages:         Annotated[list, operator.add]
    task_type:        str
    course_materials: str
    draft_output:     str
    final_output:     str
    hitl_decision:    str

lg_orchestrator_llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-pro", vertexai=True, project=PROJECT_ID,
    location="us-central1", stream_usage=True,
    model_kwargs={"thinking": {"thinking_budget": 1024}},
)
lg_worker_llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash", vertexai=True, project=PROJECT_ID,
    location="us-central1", stream_usage=True,
    model_kwargs={"thinking": {"thinking_budget": 512}},
)
print("LangGraph state schema and models ready.")

/usr/local/lib/python3.12/dist-packages/IPython/core/interactiveshell.py:3553: UserWarning: WARNING! stream_usage is not default parameter.
                stream_usage was transferred to model_kwargs.
                Please confirm that stream_usage is what you intended.
  exec(code_obj, self.user_global_ns, self.user_ns)


LangGraph state schema and models ready.


In [6]:
def router_node(state: TeacherState) -> TeacherState:
    user_msg = state["messages"][-1].content
    lower = user_msg.lower()
    if any(k in lower for k in ["email", "send", "announcement", "draft"]):
        task = "email"
    elif any(k in lower for k in ["lesson plan", "outline", "lecture"]):
        task = "lessonplan"
    elif any(k in lower for k in ["quiz", "multiple choice", "question", "mcq"]):
        task = "quiz"
    else:
        prompt = (
            "Classify into exactly one of: lessonplan / quiz / email.\n"
            f"Request: {user_msg}\nReply with ONLY the task name."
        )
        raw = lg_orchestrator_llm.invoke([HumanMessage(content=prompt)]).content.strip().lower()
        task = raw if raw in ["lessonplan", "quiz", "email"] else "lessonplan"
    print(f"[LangGraph] Router -> {task}")
    return {**state, "task_type": task}

def route_to_agent(state: TeacherState) -> Literal["lessonplanner", "quizcontent", "emaildrafter"]:
    return {"lessonplan": "lessonplanner", "quiz": "quizcontent", "email": "emaildrafter"}[state["task_type"]]

def lesson_planner_node(state: TeacherState) -> TeacherState:
    user_query = state["messages"][-1].content
    materials = retrieve_course_materials(user_query, page_size=6)
    prompt = (
        "You are an expert curriculum designer at Mae Fah Luang University.\n\n"
        f"Course Materials:\n{materials}\n\nInstructor Request:\n{user_query}\n\n"
        "Generate a comprehensive 90-minute lesson plan: Learning Objectives, "
        "timing breakdown, teaching methods, assessment strategy, required materials."
    )
    response = lg_worker_llm.invoke([HumanMessage(content=prompt)])
    return {**state, "course_materials": materials, "final_output": response.content,
            "messages": state["messages"] + [AIMessage(content=response.content)]}

def quiz_content_node(state: TeacherState) -> TeacherState:
    user_query = state["messages"][-1].content
    materials = retrieve_course_materials(user_query, page_size=8)
    prompt = (
        "You are a quiz content specialist at Mae Fah Luang University.\n\n"
        f"Course Materials:\n{materials}\n\nTopic: {user_query}\n\n"
        "Generate exactly 10 multiple-choice questions as a JSON array only.\n"
        'Each item: {"question":"...","options":["A.","B.","C.","D."],"correct_index":0,"explanation":"..."}'
    )
    response = lg_worker_llm.invoke([HumanMessage(content=prompt)])
    return {**state, "course_materials": materials, "draft_output": response.content}

def quiz_publisher_node(state: TeacherState) -> TeacherState:
    prompt = (
        f"Quiz JSON:\n{state['draft_output']}\n\n"
        "Present the quiz cleanly: number each question, list A/B/C/D options, mark correct with checkmark, include explanation."
    )
    response = lg_worker_llm.invoke([HumanMessage(content=prompt)])
    return {**state, "final_output": response.content,
            "messages": state["messages"] + [AIMessage(content=response.content)]}

def email_drafter_node(state: TeacherState) -> TeacherState:
    user_query = state["messages"][-1].content
    materials = retrieve_course_materials(user_query, page_size=2)
    prompt = (
        "You are a professional email drafting assistant for a university lecturer at Mae Fah Luang University.\n\n"
        f"Context:\n{materials}\n\nDraft a professional email based on:\n{user_query}\n\n"
        "Format EXACTLY as:\nSUBJECT: <subject line>\n\nBODY:\n<email body>\n\nUse formal university tone."
    )
    response = lg_worker_llm.invoke([HumanMessage(content=prompt)])
    return {**state, "course_materials": materials, "draft_output": response.content}

def hitl_approval_node(state: TeacherState) -> TeacherState:
    # Auto-approved for evaluation (no interactive input)
    print("[LangGraph] HITL: auto-approving for evaluation run.")
    return {**state, "hitl_decision": "approved"}

def email_sender_node(state: TeacherState) -> TeacherState:
    if state["hitl_decision"] == "approved":
        msg = f"Email approved and sent to students.\n\n{state['draft_output']}"
    else:
        msg = f"Email rejected by instructor.\n\nDraft:\n{state['draft_output']}"
    return {**state, "final_output": msg,
            "messages": state["messages"] + [AIMessage(content=msg)]}

def route_after_hitl(state: TeacherState) -> Literal["emailsender"]:
    return "emailsender"

print("All LangGraph nodes defined.")

All LangGraph nodes defined.


In [7]:
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import MemorySaver

checkpointer = MemorySaver()
builder = StateGraph(TeacherState)

builder.add_node("router",        router_node)
builder.add_node("lessonplanner", lesson_planner_node)
builder.add_node("quizcontent",   quiz_content_node)
builder.add_node("quizpublisher", quiz_publisher_node)
builder.add_node("emaildrafter",  email_drafter_node)
builder.add_node("hitlapproval",  hitl_approval_node)
builder.add_node("emailsender",   email_sender_node)

builder.set_entry_point("router")
builder.add_conditional_edges("router", route_to_agent)
builder.add_edge("lessonplanner", END)
builder.add_edge("quizcontent",   "quizpublisher")
builder.add_edge("quizpublisher", END)
builder.add_edge("emaildrafter",  "hitlapproval")
builder.add_conditional_edges("hitlapproval", route_after_hitl)
builder.add_edge("emailsender",   END)

lg_graph = builder.compile(checkpointer=checkpointer)
print("LangGraph compiled. Nodes:", list(lg_graph.get_graph().nodes.keys()))

LangGraph compiled. Nodes: ['__start__', 'router', 'lessonplanner', 'quizcontent', 'quizpublisher', 'emaildrafter', 'hitlapproval', 'emailsender', '__end__']


/usr/local/lib/python3.12/dist-packages/langgraph/cache/base/__init__.py:8: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


## 5 — Build Google ADK System

Complete replica of `02_adk_system.ipynb` with HITL auto-approved for evaluation.

In [8]:
import google.adk
from google.adk.agents import LlmAgent, SequentialAgent
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.adk.tools import FunctionTool
from google.adk.tools.agent_tool import AgentTool
from google.adk.models.google_llm import Gemini
from google.genai import types

print("ADK version:", google.adk.__version__)

resilient_http = types.HttpOptions(
    retry_options=types.HttpRetryOptions(
        attempts=5, initial_delay=5.0,
        http_status_codes=[408, 429, 500, 502, 503, 504]
    )
)
adk_pro = Gemini(
    model="gemini-2.5-pro", http_options=resilient_http,
    generate_content_config=types.GenerateContentConfig(
        thinking_config=types.ThinkingConfig(thinking_budget=0)
    ),
)
adk_flash = Gemini(
    model="gemini-2.5-flash", http_options=resilient_http,
    generate_content_config=types.GenerateContentConfig(
        thinking_config=types.ThinkingConfig(thinking_budget=0)
    ),
)
print("ADK models initialised.")

ADK version: 1.29.0
ADK models initialised.


In [9]:
# Lesson Planner Agent
adk_lesson_planner = LlmAgent(
    name="lesson_planner_agent", model=adk_flash,
    description="Creates detailed lesson plans. Use for lesson plan / course outline requests.",
    instruction="""You are an expert curriculum designer at Mae Fah Luang University.
1. Use `retrieve_course_materials` (page_size=6) to fetch relevant content.
2. Generate a 90-minute lesson plan: Learning Objectives, timing, teaching methods, assessment, materials.""",
    tools=[retrieve_course_materials]
)

# Quiz Generator (2-step Sequential)
quiz_content_agent = LlmAgent(
    name="quiz_content_agent", model=adk_flash,
    description="Retrieves course materials and generates quiz questions as JSON.",
    instruction="""You are a quiz content specialist at Mae Fah Luang University.
1. Use `retrieve_course_materials` (page_size=8).
2. Generate exactly 10 multiple-choice questions.
3. Respond ONLY with a valid JSON array.
Each item: {"question":"...","options":["A.","B.","C.","D."],"correct_index":0,"explanation":"..."}""",
    tools=[retrieve_course_materials],
    output_key="quiz_questions_json"
)
quiz_publisher_agent = LlmAgent(
    name="quiz_publisher_agent", model=adk_flash,
    description="Formats and presents the final quiz.",
    instruction="""You are a quiz publisher.
Quiz JSON: {quiz_questions_json}
Present cleanly: number each question, A/B/C/D options, mark correct, include explanation.""",
    tools=[]
)
adk_quiz_generator = SequentialAgent(
    name="quiz_generator_agent",
    description="Generates a 10-question MCQ quiz. Use for quiz/test/assessment requests.",
    sub_agents=[quiz_content_agent, quiz_publisher_agent]
)

# Email Agent (auto-approve HITL for evaluation)
email_drafter_agent = LlmAgent(
    name="email_drafter_agent", model=adk_flash,
    description="Drafts a professional email to students.",
    instruction="""You are a professional email drafting assistant for Mae Fah Luang University.
Draft a professional email based on the instructor's request.
Format EXACTLY as:
SUBJECT: <subject line>

BODY:
<email body>

Use formal university tone.""",
    output_key="email_draft",
)

def send_email_to_students(subject: str, body: str) -> dict:
    """Sends the approved email to all students.

    Args:
        subject: The email subject line.
        body: The email body content.

    Returns:
        Confirmation dict with status and recipients.
    """
    return {"status": "sent", "subject": subject, "recipients": "all_students"}

email_send_tool = FunctionTool(func=send_email_to_students)

def auto_approve_callback(tool, args, tool_context, **kwargs):
    """Auto-approve for evaluation runs."""
    print("[ADK] HITL: auto-approving for evaluation run.")
    return None  # None = proceed with actual tool

email_sender_agent = LlmAgent(
    name="email_sender_agent", model=adk_flash,
    description="Sends email to students after instructor approval.",
    instruction="""You are an email dispatch agent.
The email draft is {email_draft}.
1. Parse the draft to extract SUBJECT and BODY.
2. Call send_email_to_students with the subject and body.
3. Report the result to the instructor.""",
    tools=[email_send_tool],
    before_tool_callback=auto_approve_callback,
)
adk_email_agent = SequentialAgent(
    name="email_agent",
    description="Drafts a student email, applies HITL, then sends it. Use for email/announcement requests.",
    sub_agents=[email_drafter_agent, email_sender_agent],
)

# Root Orchestrator
adk_root = LlmAgent(
    name="teacher_assistant_orchestrator", model=adk_pro,
    description="Root AI Teaching Assistant for Mae Fah Luang University instructors.",
    instruction="""You are the AI Teaching Assistant for Mae Fah Luang University instructors.
You help with three task types: Lesson Plans, Quizzes, Emails.
For each request: identify task type, delegate to specialist agent, report full result.""",
    tools=[
        AgentTool(agent=adk_lesson_planner),
        AgentTool(agent=adk_quiz_generator),
        AgentTool(agent=adk_email_agent),
    ]
)
print("ADK root orchestrator ready.")
print("Tools:", [t.name for t in adk_root.tools])

ADK root orchestrator ready.
Tools: ['lesson_planner_agent', 'quiz_generator_agent', 'email_agent']


## 6 — Runners

In [10]:
import uuid, time, asyncio
from langchain_core.callbacks.usage import UsageMetadataCallbackHandler
from langsmith import traceable
from langsmith.run_helpers import get_current_run_tree

USERID = "mfu-instructor-01"

@traceable(name="lg_run_request", run_type="chain",
           metadata={"framework": "LangGraph", "app": "teacher-assistant-langgraph"})
def lg_run(user_input: str, scenario: str = "unknown"):
    thread_id = str(uuid.uuid4())
    cb = UsageMetadataCallbackHandler()
    config = {
        "configurable": {"thread_id": thread_id},
        "callbacks": [cb],
        "metadata": {"scenario": scenario, "framework": "LangGraph"},
    }
    init_state = {
        "messages": [HumanMessage(content=user_input)],
        "task_type": "", "course_materials": "",
        "draft_output": "", "final_output": "", "hitl_decision": "",
    }
    start = time.time()
    result = lg_graph.invoke(init_state, config)
    latency = round(time.time() - start, 2)

    total_input  = sum(m.get("input_tokens",  0) for m in cb.usage_metadata.values())
    total_output = sum(m.get("output_tokens", 0) for m in cb.usage_metadata.values())
    total_thought = sum(
        m.get("output_token_details", {}).get("reasoning", 0)
        for m in cb.usage_metadata.values()
    )
    try:
        run_tree = get_current_run_tree()
        ls_run_id = str(run_tree.id) if run_tree else None
        if run_tree:
            run_tree.add_metadata({"input_tokens": total_input, "output_tokens": total_output,
                                    "thought_tokens": total_thought})
    except Exception:
        ls_run_id = None

    print(f"  [LangGraph] {latency}s | in={total_input} think={total_thought} out={total_output}")
    return result.get("final_output", ""), latency, total_input, total_output, total_thought, ls_run_id

print("LangGraph runner ready.")

LangGraph runner ready.


In [11]:
ADK_APP_NAME = "teacher-assistant-adk"
adk_session_service = InMemorySessionService()
adk_runner = Runner(
    agent=adk_root, app_name=ADK_APP_NAME, session_service=adk_session_service,
)

@traceable(name="adk_run_request", run_type="chain",
           metadata={"framework": "Google ADK", "app": "teacher-assistant-adk"})
async def adk_run(user_input: str, scenario: str = "unknown"):
    session_id = str(uuid.uuid4())
    await adk_session_service.create_session(
        app_name=ADK_APP_NAME, user_id=USERID, session_id=session_id,
    )
    message = types.Content(role="user", parts=[types.Part(text=user_input)])
    run_tree = get_current_run_tree()
    ls_run_id = str(run_tree.id) if run_tree else None

    start = time.time()
    final_response = ""
    total_input = total_output = total_thought = event_count = 0

    async for event in adk_runner.run_async(
        user_id=USERID, session_id=session_id, new_message=message,
    ):
        if hasattr(event, "usage_metadata") and event.usage_metadata:
            total_input   += getattr(event.usage_metadata, "prompt_token_count",    0) or 0
            total_output  += getattr(event.usage_metadata, "candidates_token_count",0) or 0
            total_thought += getattr(event.usage_metadata, "thoughts_token_count",  0) or 0
            event_count   += 1
        if event.is_final_response() and event.content:
            final_response = event.content.parts[0].text.strip()

    latency = round(time.time() - start, 2)
    print(f"  [ADK] {latency}s | {event_count} events | in={total_input} think={total_thought} out={total_output}")
    return final_response, latency, total_input, total_output, total_thought, ls_run_id

print("ADK runner ready.")

ADK runner ready.


## 7 — Evaluation Scenarios

Set `N_RUNS = 1` for a quick single-pass judge evaluation.  
Increase to `3` or `5` for statistical averages (takes proportionally longer).

In [12]:
SCENARIOS = {
    "lesson_plan": "Create a 90-minute lesson plan for week 1 on Software Testing for second-year Software Engineering students. Align it with the course materials.",
    "quiz":        "Generate 10 multiple-choice questions on Software Testing from the course materials.",
    "email":       "Draft and send an email to all students reminding them that the Software Testing quiz covering Unit Testing and Black Box Testing is next Monday at 9am. Include what topics to study.",
}

N_RUNS = 3  # Increase to 3-5 for statistical averages

print("Scenarios:")
for k, v in SCENARIOS.items():
    print(f"  [{k}] {v[:80]}...")
print(f"\nN_RUNS = {N_RUNS} per scenario per framework")

Scenarios:
  [lesson_plan] Create a 90-minute lesson plan for week 1 on Software Testing for second-year So...
  [quiz] Generate 10 multiple-choice questions on Software Testing from the course materi...
  [email] Draft and send an email to all students reminding them that the Software Testing...

N_RUNS = 3 per scenario per framework


In [13]:
eval_results = {}  # { scenario_key: { "lg": [...], "adk": [...] } }

for scenario_key, prompt in SCENARIOS.items():
    print(f"\n{'='*60}")
    print(f"Scenario: {scenario_key}")
    print("="*60)
    eval_results[scenario_key] = {"lg": [], "adk": []}

    for run_i in range(N_RUNS):
        print(f"\n  Run {run_i+1}/{N_RUNS}")

        # LangGraph
        print("  [LangGraph] running...")
        try:
            lg_resp, lg_lat, lg_in, lg_out, lg_think, lg_lsid = lg_run(
                user_input=prompt, scenario=f"{scenario_key}_lg"
            )
            eval_results[scenario_key]["lg"].append({
                "response": lg_resp, "latency": lg_lat,
                "input_tokens": lg_in, "output_tokens": lg_out,
                "thought_tokens": lg_think, "error": None
            })
        except Exception as e:
            eval_results[scenario_key]["lg"].append({"response": "", "latency": 0, "error": str(e)})
            print(f"  [LangGraph] ERROR: {e}")

        # Buffer between frameworks
        await asyncio.sleep(15)

        # ADK
        print("  [ADK] running...")
        try:
            adk_resp, adk_lat, adk_in, adk_out, adk_think, adk_lsid = await adk_run(
                user_input=prompt, scenario=f"{scenario_key}_adk"
            )
            eval_results[scenario_key]["adk"].append({
                "response": adk_resp, "latency": adk_lat,
                "input_tokens": adk_in, "output_tokens": adk_out,
                "thought_tokens": adk_think, "error": None
            })
        except Exception as e:
            eval_results[scenario_key]["adk"].append({"response": "", "latency": 0, "error": str(e)})
            print(f"  [ADK] ERROR: {e}")

        if run_i < N_RUNS - 1:
            print("  Waiting 30s before next run...")
            await asyncio.sleep(30)

print("\nAll scenarios completed.")


Scenario: lesson_plan

  Run 1/3
  [LangGraph] running...
[LangGraph] Router -> lessonplan
  [LangGraph] 23.32s | in=171 think=2046 out=4092
  [ADK] running...


  [ADK] 24.95s | 2 events | in=1555 think=237 out=1002
  Waiting 30s before next run...

  Run 2/3
  [LangGraph] running...
[LangGraph] Router -> lessonplan
  [LangGraph] 24.62s | in=171 think=2604 out=4092
  [ADK] running...
  [ADK] 23.94s | 2 events | in=1496 think=89 out=1001
  Waiting 30s before next run...

  Run 3/3
  [LangGraph] running...
[LangGraph] Router -> lessonplan
  [LangGraph] 22.86s | in=171 think=1258 out=4092
  [ADK] running...
  [ADK] 24.79s | 2 events | in=1645 think=232 out=1007

Scenario: quiz

  Run 1/3
  [LangGraph] running...
[LangGraph] Router -> quiz
  [LangGraph] 23.47s | in=1455 think=2053 out=4347
  [ADK] running...
  [ADK] 29.84s | 2 events | in=1488 think=134 out=1021
  Waiting 30s before next run...

  Run 2/3
  [LangGraph] running...
[LangGraph] Router -> quiz
  [LangGraph] 25.39s | in=1312 think=2244 out=4173
  [ADK] running...
  [ADK] 28.93s | 2 events | in=1544 think=144 out=1081
  Waiting 30s before next run...

  Run 3/3
  [LangGraph] running...


## 8 — Judge LLM (Gemini 2.5 Pro)

| Dimension | What is scored |
|---|---|
| **accuracy** | Factual correctness, no hallucinations |
| **completeness** | All required elements present |
| **clarity** | Well-structured, readable, formatted |
| **edu_relevance** | Appropriate for MFU educational context |
| **task_adherence** | Followed the exact task instructions |

In [14]:
import json, re

judge_llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-pro", vertexai=True, project=PROJECT_ID,
    location="us-central1", max_output_tokens=2048,
    model_kwargs={"thinking": {"thinking_budget": 2048}},
)

RUBRIC = RUBRIC = """
You are an expert educational AI evaluator. Score the following AI-generated response
on a scale of 1-5 for each dimension. Be rigorous and critical.


## Special Evaluation Notes
- Email scenario: the system uses a mock send_email_to_students() tool that returns
  {"status": "sent"} — this is intentional test scaffolding, not a real API call.
  If the response reports the email was sent, treat this as CORRECT task completion.
  Do NOT penalize accuracy or task_adherence for this.
- Email scenario: details like date, time, and topics (e.g., "Monday 9am",
  "Unit Testing", "Black Box Testing") come from the instructor's prompt itself.
  These are NOT hallucinations — do NOT penalize accuracy for repeating them.


Dimensions:
- accuracy:        Factual correctness, no hallucinations (1=wrong, 5=fully accurate)
- completeness:    Covers ALL required elements (1=missing most, 5=fully complete)
- clarity:         Structure, readability, professional formatting (1=hard to follow, 5=excellent)
- edu_relevance:   Appropriate for Mae Fah Luang University context (1=generic, 5=perfectly tailored)
- task_adherence:  Followed exact task instructions (1=ignored, 5=perfect compliance)


Respond ONLY with valid JSON (no markdown, no extra text):
{
  "accuracy":       <1-5>,
  "completeness":   <1-5>,
  "clarity":        <1-5>,
  "edu_relevance":  <1-5>,
  "task_adherence": <1-5>,
  "reasoning":      "<1-2 sentence justification>"
}
"""

@traceable(name="llm_judge", run_type="llm",
           metadata={"model": "gemini-2.5-pro", "role": "judge"})
def judge_response(prompt: str, response: str, framework: str, scenario: str) -> dict:
    """Score one response with the LLM judge."""
    if not response or not response.strip():
        return {"accuracy":0,"completeness":0,"clarity":0,"edu_relevance":0,
                "task_adherence":0,"reasoning":"Empty response.",
                "total":0,"framework":framework,"scenario":scenario}

    content = (
        f"TASK: {scenario}\n\nPROMPT:\n{prompt}\n\nFRAMEWORK: {framework}\n\n"
        f"RESPONSE (truncated to 3000 chars):\n{response[:3000]}\n\n{RUBRIC}"
    )
    raw = judge_llm.invoke([HumanMessage(content=content)]).content.strip()
    raw = re.sub(r"^```(?:json)?\s*", "", raw)
    raw = re.sub(r"\s*```$", "", raw)

    try:
        scores = json.loads(raw)
    except Exception:
        print(f"  [Judge] JSON parse failed. Raw: {raw[:200]}")
        scores = {"accuracy":1,"completeness":1,"clarity":1,"edu_relevance":1,
                  "task_adherence":1,"reasoning":"Parse error."}

    scores["total"] = (
        scores.get("accuracy",0) + scores.get("completeness",0) +
        scores.get("clarity",0) + scores.get("edu_relevance",0) +
        scores.get("task_adherence",0)
    )
    scores["framework"] = framework
    scores["scenario"]  = scenario
    return scores

print("Judge LLM ready (Gemini 2.5 Pro, thinking_budget=2048).")

Judge LLM ready (Gemini 2.5 Pro, thinking_budget=2048).


## 9 — Score All Responses

In [15]:
all_scores = []

for scenario_key, prompt in SCENARIOS.items():
    print(f"\n{'='*60}")
    print(f"Judging: {scenario_key}")
    print("="*60)

    for framework_key, fw_label in [("lg", "LangGraph"), ("adk", "Google ADK")]:
        runs = eval_results[scenario_key][framework_key]
        for run_i, run_data in enumerate(runs):
            print(f"  [{fw_label}] run {run_i+1}...", end=" ")
            scores = judge_response(
                prompt=prompt,
                response=run_data.get("response", ""),
                framework=fw_label,
                scenario=scenario_key,
            )
            scores["run"]           = run_i + 1
            scores["latency"]       = run_data.get("latency", 0)
            scores["input_tokens"]  = run_data.get("input_tokens",  0)
            scores["output_tokens"] = run_data.get("output_tokens", 0)
            scores["thought_tokens"]= run_data.get("thought_tokens",0)
            all_scores.append(scores)
            print(f"total={scores['total']}/25 | {scores.get('reasoning','')[:70]}")
        await asyncio.sleep(5)

print("\nAll judging complete.")


Judging: lesson_plan
  [LangGraph] run 1... total=21/25 | The response is well-structured, accurate, and tailored to the specifi
  [LangGraph] run 2... total=21/25 | The response is well-structured, accurate, and perfectly tailored to t
  [LangGraph] run 3... total=24/25 | The lesson plan is professional, well-structured, and perfectly aligns
  [Google ADK] run 1... total=24/25 | The lesson plan is excellent—accurate, complete, and perfectly structu
  [Google ADK] run 2... total=24/25 | The response is a well-structured, clear, and accurate lesson plan per
  [Google ADK] run 3... total=19/25 | The response is a well-structured and accurate lesson plan, but it com

Judging: quiz
  [LangGraph] run 1... total=20/25 | The response is accurate and clearly formatted, but incomplete as it o
  [LangGraph] run 2... total=19/25 | The response is accurate and clearly formatted but incomplete, providi
  [LangGraph] run 3... total=18/25 | The response is incomplete, providing only 8 full questions

CancelledError: 

## 10 — Aggregate Results

In [ ]:
import pandas as pd

df_scores = pd.DataFrame(all_scores)

# Per-scenario summary
summary = df_scores.groupby(["framework","scenario"]).agg(
    runs           =("total",         "count"),
    avg_total      =("total",         "mean"),
    accuracy       =("accuracy",      "mean"),
    completeness   =("completeness",  "mean"),
    clarity        =("clarity",       "mean"),
    edu_relevance  =("edu_relevance", "mean"),
    task_adherence =("task_adherence","mean"),
    avg_latency    =("latency",       "mean"),
    avg_input_tok  =("input_tokens",  "mean"),
    avg_output_tok =("output_tokens", "mean"),
).round(2)

print("=" * 70)
print("  LLM-as-a-Judge Scores (out of 25 total, 5 per dimension)")
print("=" * 70)
print(summary.to_string())

overall = df_scores.groupby("framework").agg(
    avg_total      =("total",         "mean"),
    accuracy       =("accuracy",      "mean"),
    completeness   =("completeness",  "mean"),
    clarity        =("clarity",       "mean"),
    edu_relevance  =("edu_relevance", "mean"),
    task_adherence =("task_adherence","mean"),
    avg_latency    =("latency",       "mean"),
).round(2)

print("\n" + "=" * 70)
print("  Overall Framework Comparison")
print("=" * 70)
print(overall.to_string())

## 11 — Visualisations

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

dims          = ["accuracy","completeness","clarity","edu_relevance","task_adherence"]
scenarios_list = list(SCENARIOS.keys())
frameworks     = ["LangGraph", "Google ADK"]
colors         = {"LangGraph": "#4C72B0", "Google ADK": "#DD8452"}

# Chart 1: Stacked bar per scenario
fig, axes = plt.subplots(1, len(scenarios_list), figsize=(14, 5), sharey=True)
fig.suptitle("LLM Judge Scores by Scenario & Dimension (out of 25)", fontsize=13, fontweight="bold")

for ax, scenario_key in zip(axes, scenarios_list):
    x = np.arange(len(frameworks))
    bottoms = np.zeros(len(frameworks))
    dim_colors = plt.cm.tab10(np.linspace(0, 0.5, len(dims)))
    for dim, col in zip(dims, dim_colors):
        vals = []
        for fw in frameworks:
            sub = df_scores[(df_scores["framework"]==fw) & (df_scores["scenario"]==scenario_key)]
            vals.append(sub[dim].mean() if len(sub) > 0 else 0)
        ax.bar(x, vals, bottom=bottoms, color=col, label=dim, width=0.5)
        bottoms += np.array(vals)
    ax.set_xticks(x)
    ax.set_xticklabels([fw.replace(" ","\n") for fw in frameworks], fontsize=9)
    ax.set_title(scenario_key.replace("_"," ").title(), fontsize=10)
    ax.set_ylim(0, 27)
    ax.grid(axis="y", linestyle="--", alpha=0.4)

axes[0].set_ylabel("Score (0-25)", fontsize=10)
handles = [mpatches.Patch(color=c, label=d) for d, c in zip(dims, dim_colors)]
fig.legend(handles=handles, loc="lower center", ncol=5, fontsize=9, bbox_to_anchor=(0.5, -0.08))
plt.tight_layout()
plt.savefig("judge_scores_by_scenario.png", dpi=150, bbox_inches="tight")
plt.show()
print("Chart 1 saved: judge_scores_by_scenario.png")

In [ ]:
# Chart 2: Radar chart
N = len(dims)
angles = np.linspace(0, 2*np.pi, N, endpoint=False).tolist()
angles += angles[:1]

fig, ax = plt.subplots(figsize=(7, 7), subplot_kw={"polar": True})
ax.set_title("Overall Quality Radar\n(LangGraph vs Google ADK)", fontsize=12, fontweight="bold", pad=20)

for fw in frameworks:
    sub = df_scores[df_scores["framework"] == fw]
    vals = [sub[d].mean() for d in dims] + [sub[dims[0]].mean()]
    ax.plot(angles, vals, "o-", linewidth=2, color=colors[fw], label=fw)
    ax.fill(angles, vals, alpha=0.1, color=colors[fw])

ax.set_xticks(angles[:-1])
ax.set_xticklabels([d.replace("_"," ").title() for d in dims], fontsize=10)
ax.set_ylim(0, 5)
ax.set_yticks([1,2,3,4,5])
ax.legend(loc="upper right", bbox_to_anchor=(1.3, 1.1), fontsize=11)
plt.tight_layout()
plt.savefig("judge_radar.png", dpi=150, bbox_inches="tight")
plt.show()
print("Chart 2 saved: judge_radar.png")

In [ ]:
# Chart 3: Scatter — quality vs latency
fig, ax = plt.subplots(figsize=(8, 5))
ax.set_title("Quality Score vs Latency\n(bubble size = thought tokens)", fontsize=12, fontweight="bold")

for fw in frameworks:
    sub = df_scores[df_scores["framework"] == fw]
    x = sub["latency"].values
    y = sub["total"].values
    s = np.clip(sub["thought_tokens"].values / 20, 20, 500)
    ax.scatter(x, y, s=s, alpha=0.6, color=colors[fw], label=fw)

ax.set_xlabel("Latency (s)", fontsize=11)
ax.set_ylabel("Judge Total Score (0-25)", fontsize=11)
ax.set_ylim(0, 27)
ax.grid(linestyle="--", alpha=0.4)
ax.legend(fontsize=11)
plt.tight_layout()
plt.savefig("judge_scatter.png", dpi=150)
plt.show()
print("Chart 3 saved: judge_scatter.png")

## 12 — LLM Verdict

In [ ]:
@traceable(name="judge_verdict", run_type="llm",
           metadata={"model": "gemini-2.5-pro", "role": "verdict"})
def generate_verdict(summary_df: pd.DataFrame) -> str:
    """Ask the judge LLM to render a final verdict backed by scores."""
    verdict_prompt = f"""
You are an expert AI researcher evaluating two multi-agent AI Teaching Assistant systems
for Mae Fah Luang University, Thailand.

System 1: LangGraph (stateful graph-based orchestration)
System 2: Google ADK (agent tool delegation)

Aggregated LLM-as-a-Judge scores (each dimension 0-5, total 0-25):
{summary_df.to_string()}

Write a concise verdict (max 250 words) covering:
1. Which framework performed better overall and why
2. Where each framework has a clear advantage
3. Trade-offs (latency vs quality)
4. Practical recommendation for an educational AI assistant in Myanmar/Thailand

Be specific and data-driven. Reference actual scores.
"""
    return judge_llm.invoke([HumanMessage(content=verdict_prompt)]).content.strip()

verdict_text = generate_verdict(overall)
print("\n" + "="*70)
print("  FINAL VERDICT")
print("="*70)
print(verdict_text)

## 13 — Export Results

In [ ]:
from datetime import datetime, timezone
from google.colab import files

# Save scores CSV
scores_csv = "judge_results.csv"
df_scores.to_csv(scores_csv, index=False)
print(f"Scores saved: {scores_csv} ({len(df_scores)} rows)")

# Save markdown report
report_md = "judge_report.md"
with open(report_md, "w") as f:
    f.write(f"# LLM-as-a-Judge Evaluation Report\n")
    f.write(f"Generated: {datetime.now(timezone.utc).isoformat()}\n\n")
    f.write("## Overall Framework Comparison\n\n")
    f.write("```\n" + overall.to_string() + "\n```\n\n")
    f.write("## Per-Scenario Breakdown\n\n")
    f.write("```\n" + summary.to_string() + "\n```\n\n")
    f.write("## Final Verdict\n\n")
    f.write(verdict_text + "\n\n")
    f.write("## Individual Reasoning\n\n")
    for _, row in df_scores.iterrows():
        f.write(f"### {row['framework']} | {row['scenario']} (Run {row['run']})\n")
        f.write(f"- **Total:** {row['total']}/25 | Latency: {row['latency']}s\n")
        f.write(f"- **Reasoning:** {row.get('reasoning','')}\n\n")

print(f"Report saved: {report_md}")

files.download(scores_csv)
files.download(report_md)
print("\nDownloads triggered.")